### Culturomics  Method 3: Collective Memory
**Replication of Michel (2011) Approach on Wikipedia Revision History (2001–2024)**
**Course:** Computational Linguistics, Ca' Foscari University of Venice
**Original paper:** Michel, J.-B.(2011). *Quantitative Analysis of Culture Using Millions of Digitized Books*. Science, 331(6014), 176–182.

---

#### What This Notebook Does

Michel et al. (2011) measured **collective memory** by tracking how often specific historical years (e.g. "1883", "1910", "1950") were mentioned in books written after those years.
They found a characteristic decay curve: a sharp peak immediately after the event, followed by gradual forgetting. Crucially, they showed that this forgetting is accelerating across generations "1880" lost half its peak value in 32 years, "1973" in only 10 years.

This notebook applies the same method to Wikipedia revision snapshots (2001–2024), asking:
> Does collective memory behave differently in a **living digital document** compared to a static printed book corpus? Does Wikipedia forget — or does it continuously revise?

**The range is analogue of Michel's(2011) from 1875 to 1975:**
Tracked 14 "anchor years", each corresponding to a historically significant event:    
`1900, 1914, 1918, 1929, 1939, 1945, 1963, 1969, 1989, 2001, 2003, 2008, 2011, 2016`

**Key hypotheses:**
1. *Replication*: Wikipedia shows the same peak-then-decay pattern as Google Books
2. *Divergence*: Wikipedia as a living document may **not** decay  editors keep revising, so old dates may remain stable rather than disappearing
3. *Acceleration*: If decay exists, it may be faster than in print (consistent with Michel's (2011) finding that forgetting accelerates with each generation)


## 0. Setup

In [ ]:
import requests
import time
import re
import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit
from scipy.ndimage import uniform_filter1d

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#FAFAFA',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    10,
    'axes.labelsize':    8.5,
    'xtick.labelsize':   7.5,
    'ytick.labelsize':   7.5,
})

PRIMARY   = '#C0392B'
SECONDARY = '#2C3E50'
ACCENT    = '#E67E22'
GREEN     = '#27AE60'
PURPLE    = '#8E44AD'

print("Setup complete.")

## 1. Configuration

### Target Years (our analogue of Michel et al.'s 1875–1975)

Michel et al. tracked 100 years with annual granularity.
We track 14 historically significant "anchor years" that span from
the early 20th century into the Wikipedia era (2001–2024).

Each year is annotated with its primary historical association —
this is important for the qualitative interpretation in your essay.


In [ ]:
# ─── PASTE YOUR ARTICLE LIST FROM METHOD 1 HERE ───────────────
# (same list as in the Method 1 notebook)
ARTICLES = [
    "Albert_Einstein",
    "Charles_Darwin",
    "World_War_II",
    "Photosynthesis",
    "Climate_change",
    "Artificial_intelligence",
    "Leonardo_da_Vinci",
    "DNA",
    "French_Revolution",
    "Black_hole",
    # ADD YOUR FULL 80-ARTICLE LIST HERE
]

# ─── TARGET YEARS — analogue of Michel et al.'s 1875–1975 ─────
# Format: year → short label for plots
TARGET_YEARS = {
    1900: "1900 (turn of century)",
    1914: "1914 (WWI begins)",
    1918: "1918 (WWI ends)",
    1929: "1929 (Great Depression)",
    1939: "1939 (WWII begins)",
    1945: "1945 (WWII ends)",
    1963: "1963 (JFK assassination)",
    1969: "1969 (Moon landing)",
    1989: "1989 (Berlin Wall / Cold War end)",
    2001: "2001 (9/11)",
    2003: "2003 (Iraq War begins)",
    2008: "2008 (Financial Crisis)",
    2011: "2011 (Arab Spring / Occupy)",
    2016: "2016 (Brexit / US election)",
}

# Observation window: Wikipedia snapshots 2001–2024
OBS_YEARS  = list(range(2001, 2025))

# Cache
OUTPUT_DIR = Path("culturomics_method3_output")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_FILE = Path("culturomics_method1_output/corpus_cache.json")  # reuse Method 1 cache!
ALT_CACHE  = OUTPUT_DIR / "corpus_cache_m3.json"                  # fallback if M1 not run

API_URL = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "CulturomicsEssayBot/1.0 (academic research)"}

print(f"Articles loaded:  {len(ARTICLES)}")
print(f"Target years:     {list(TARGET_YEARS.keys())}")
print(f"Observation range: {OBS_YEARS[0]}–{OBS_YEARS[-1]}")

## 2. Load Corpus

We **reuse the corpus already fetched in Method 1** (same Wikipedia snapshots,
same articles, same years) — no additional API calls needed.

If you haven't run Method 1 yet, this cell will attempt to fetch the corpus fresh.


In [ ]:
def clean_wikitext(raw: str) -> str:
    text = re.sub(r'\{\{[^}]*\}\}', ' ', raw)
    text = re.sub(r'\[\[(?:File|Image):[^\]]*\]\]', ' ', text, flags=re.I)
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]+)\]\]', r'\1', text)
    text = re.sub(r'\[https?://\S+[^\]]*\]', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'={2,}[^=]+=+', ' ', text)
    text = re.sub(r"'{2,}", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def fetch_yearly_snapshot(article, year):
    params = {
        "action": "query", "titles": article, "prop": "revisions",
        "rvprop": "ids|timestamp", "rvlimit": 1,
        "rvstart": f"{year}-01-20T00:00:00Z",
        "rvend":   f"{year}-01-01T00:00:00Z",
        "rvdir":   "older", "format": "json",
    }
    try:
        r = requests.get(API_URL, params=params, headers=HEADERS, timeout=20)
        pages = r.json().get("query", {}).get("pages", {})
        page  = list(pages.values())[0]
        revs  = page.get("revisions", [])
        if not revs: return ""
        rev_id = revs[0]["revid"]
        r2 = requests.get(API_URL, headers=HEADERS, timeout=30, params={
            "action": "query", "revids": rev_id, "prop": "revisions",
            "rvprop": "content", "rvslots": "main", "format": "json",
        })
        pages2 = r2.json().get("query", {}).get("pages", {})
        raw = list(pages2.values())[0]["revisions"][0]["slots"]["main"]["*"]
        return clean_wikitext(raw)
    except Exception:
        return ""

# Try loading from Method 1 cache first
cache_path = CACHE_FILE if CACHE_FILE.exists() else ALT_CACHE
if cache_path.exists():
    with open(cache_path, encoding='utf-8') as f:
        corpus = json.load(f)
    print(f"Loaded corpus from cache: {cache_path}")
    print(f"Articles in cache: {len(corpus)}")
    print(f"Non-empty snapshots: {sum(1 for a in corpus for y in corpus[a] if corpus[a][y])}")
else:
    print("No cache found. Fetching corpus from Wikipedia API...")
    corpus = {}
    total = len(ARTICLES) * len(OBS_YEARS)
    done  = 0
    for article in ARTICLES:
        corpus[article] = {}
        for year in OBS_YEARS:
            corpus[article][str(year)] = fetch_yearly_snapshot(article, year)
            done += 1
            if done % 20 == 0:
                print(f"  {done}/{total} ({done/total*100:.0f}%)")
                with open(ALT_CACHE, 'w') as f:
                    json.dump(corpus, f, ensure_ascii=False)
            time.sleep(0.4)
    with open(ALT_CACHE, 'w') as f:
        json.dump(corpus, f, ensure_ascii=False)
    print("Corpus fetched and saved.")

## 3. Compute Collective Memory Frequencies

For each **observation year** (when the Wikipedia snapshot was taken, 2001–2024)
and each **target year** (the historical event we are measuring memory of):

```
freq(target_year, obs_year) = count("target_year" as string in text) / total_tokens
```

This exactly mirrors Michel et al.'s method of counting the 1-gram "1951" in books
published after 1951 to measure how long that year remained in collective memory.

**Important:** we only count a target year in observation years that come *after* it —
you cannot "remember" something that hasn't happened yet.


In [ ]:
def tokenize(text):
    return re.findall(r'\b[a-z0-9]+\b', text.lower())

def count_year_in_text(text, target_year):
    """Count how many times the string 'target_year' (as 4-digit integer)
    appears in text. Uses word-boundary regex to avoid false positives
    (e.g. '19451' should not count as '1945')."""
    return len(re.findall(r'\b' + str(target_year) + r'\b', text))


# Build the memory frequency table
rows = []

for obs_year in OBS_YEARS:
    # Aggregate text from all articles for this observation year
    combined_text = ""
    n_articles_found = 0
    for article in ARTICLES:
        text = corpus.get(article, {}).get(str(obs_year), "")
        if text:
            combined_text += " " + text
            n_articles_found += 1

    total_tokens = len(tokenize(combined_text)) if combined_text else 0

    for target_year in TARGET_YEARS:
        # Only measure memory of years that have already passed
        if target_year >= obs_year:
            continue

        raw_count = count_year_in_text(combined_text, target_year)
        rel_freq  = raw_count / total_tokens if total_tokens > 0 else 0
        lag       = obs_year - target_year   # years since the event

        rows.append({
            "obs_year":    obs_year,
            "target_year": target_year,
            "raw_count":   raw_count,
            "rel_freq":    rel_freq,
            "rel_freq_10k": rel_freq * 10_000,  # per 10,000 tokens (more readable)
            "lag":         lag,
            "n_articles":  n_articles_found,
            "total_tokens": total_tokens,
        })

df_mem = pd.DataFrame(rows)
print(f"Memory table shape: {df_mem.shape}")
print(f"\nSample rows:")
df_mem[df_mem['target_year'] == 1945].head(8)

## 4. Half-Life Calculation

Michel et al.'s key finding was that "1880" declined to half its peak value
within 32 years, while "1973" declined to half within only 10 years —
demonstrating **accelerating forgetting**.

We compute the same metric: for each target year, we find the peak frequency
(the year it was most mentioned in our corpus), then measure how long until
frequency drops to 50% of that peak.

**Important caveat for the essay:**
In a printed corpus, a book written in 1920 does not change — so the frequency
of "1883" in 1920's books reflects genuine collective attention of that moment.
In Wikipedia, the same article is *continuously updated* — so a drop in frequency
might reflect **active editorial choice** (editors removing the date from the text)
rather than passive societal forgetting. This distinction is central to our
"and so what" argument.


In [ ]:
half_life_results = []

for target_year, label in TARGET_YEARS.items():
    sub = df_mem[df_mem['target_year'] == target_year].copy()
    if sub.empty or sub['rel_freq_10k'].max() == 0:
        continue

    # Peak: highest frequency observation
    peak_idx  = sub['rel_freq_10k'].idxmax()
    peak_freq = sub.loc[peak_idx, 'rel_freq_10k']
    peak_year = sub.loc[peak_idx, 'obs_year']
    half_val  = peak_freq / 2

    # Find the first observation year AFTER the peak where freq <= half_val
    post_peak = sub[sub['obs_year'] > peak_year].copy()
    below_half = post_peak[post_peak['rel_freq_10k'] <= half_val]

    if not below_half.empty:
        half_life_year = below_half.iloc[0]['obs_year']
        half_life      = half_life_year - peak_year
    else:
        half_life_year = None
        half_life      = None  # didn't drop below half in our window

    half_life_results.append({
        "target_year":    target_year,
        "label":          label,
        "peak_freq_10k":  round(peak_freq, 4),
        "peak_obs_year":  peak_year,
        "lag_to_peak":    peak_year - target_year,
        "half_life_yrs":  half_life,
        "half_life_year": half_life_year,
    })

df_hl = pd.DataFrame(half_life_results)
print(df_hl[['target_year', 'label', 'peak_freq_10k',
             'peak_obs_year', 'lag_to_peak', 'half_life_yrs']].to_string())

## 5. Direct Comparison with Michel et al. (2011)

We replicate Table 1 (implicit) from Michel et al., adding our Wikipedia findings.


In [ ]:
# Michel et al.'s key reported values
michel_data = {
    "1880": {"half_life_books": 32, "medium": "Google Books (print)"},
    "1973": {"half_life_books": 10, "medium": "Google Books (print)"},
}

print("=" * 65)
print("COLLECTIVE MEMORY: HALF-LIFE COMPARISON")
print("Michel et al. (2011) Google Books  vs.  Wikipedia (this study)")
print("=" * 65)
print(f"{'Year':<8} {'Peak (Wikipedia)':<22} {'Half-life (Wikipedia)':<25} {'Michel et al. (Books)'}")
print("-" * 65)

for _, row in df_hl.iterrows():
    ty   = int(row['target_year'])
    peak = f"obs {int(row['peak_obs_year'])} ({row['peak_freq_10k']:.3f}/10k)"
    hl   = f"{int(row['half_life_yrs'])} yrs" if row['half_life_yrs'] else "not reached"
    comp = "~32 yrs" if ty == 1880 else ("~10 yrs" if ty == 1973 else "N/A")
    print(f"{ty:<8} {peak:<22} {hl:<25} {comp}")

print()
print("Interpretation note:")
print("  'not reached' = frequency never dropped below 50% of peak in our 2001-2024 window.")
print("  This may indicate Wikipedia's LIVING DOCUMENT nature: editors keep dates in the text.")

## 6. Visualisation

### Panel A — Memory Curves (mirrors Michel et al. Fig. 3A)
### Panel B — Half-life comparison (our original contribution)
### Panel C — Peak lag: how long after an event does Wikipedia most discuss it?
### Panel D — Cross-target heatmap: which years are remembered most, by whom


In [ ]:
import matplotlib.cm as cm

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)

# Colour map: one colour per target year
target_years_list = list(TARGET_YEARS.keys())
cmap   = cm.get_cmap('tab20', len(target_years_list))
color_for = {ty: cmap(i) for i, ty in enumerate(target_years_list)}

# ── Panel A: Memory curves ───────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])

for ty, label in TARGET_YEARS.items():
    sub = df_mem[df_mem['target_year'] == ty].copy()
    if sub.empty or sub['rel_freq_10k'].max() == 0:
        continue
    col = color_for[ty]
    ax1.plot(sub['obs_year'], sub['rel_freq_10k'],
             lw=1.8, color=col, alpha=0.85, label=str(ty))
    # Mark peak
    peak_row = sub.loc[sub['rel_freq_10k'].idxmax()]
    ax1.scatter([peak_row['obs_year']], [peak_row['rel_freq_10k']],
                color=col, s=40, zorder=4)

ax1.set_title("A. Collective Memory Curves\n(frequency of historical year-strings, per 10k tokens)",
              fontweight='bold')
ax1.set_xlabel("Wikipedia snapshot year (observation)")
ax1.set_ylabel("Frequency per 10,000 tokens")
ax1.legend(fontsize=6.5, ncol=2, loc='upper right', title="Target year", title_fontsize=7)

# Michel et al. reference annotation
ax1.text(0.02, 0.97,
         "Michel et al. (2011) found:
'1880' loses half its peak in 32 yrs
'1973' loses half in 10 yrs",
         transform=ax1.transAxes, fontsize=7, va='top', ha='left',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# ── Panel B: Half-life comparison bar chart ───────────────────
ax2 = fig.add_subplot(gs[0, 1])
df_hl_valid = df_hl.dropna(subset=['half_life_yrs']).copy()
df_hl_valid = df_hl_valid.sort_values('target_year')

bars = ax2.bar(df_hl_valid['target_year'].astype(str),
               df_hl_valid['half_life_yrs'],
               color=[color_for[ty] for ty in df_hl_valid['target_year']],
               alpha=0.8, width=0.6)

# Reference lines from Michel et al.
ax2.axhline(32, color=SECONDARY, lw=1.5, ls='--', alpha=0.6,
            label="Michel et al.: '1880' half-life = 32 yrs")
ax2.axhline(10, color=ACCENT, lw=1.5, ls='--', alpha=0.6,
            label="Michel et al.: '1973' half-life = 10 yrs")

for bar in bars:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 0.3,
             f'{int(h)}y', ha='center', fontsize=7, fontweight='bold')

ax2.set_title("B. Half-Life of Collective Memory in Wikipedia\nvs. Michel et al. reference values",
              fontweight='bold')
ax2.set_xlabel("Target year")
ax2.set_ylabel("Years until frequency drops to 50% of peak")
ax2.legend(fontsize=7.5, loc='upper right')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')

# ── Panel C: Lag to peak ─────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
df_hl_sorted = df_hl.sort_values('target_year')
colors_c = [color_for[ty] for ty in df_hl_sorted['target_year']]

ax3.scatter(df_hl_sorted['target_year'], df_hl_sorted['lag_to_peak'],
            c=colors_c, s=120, zorder=3, alpha=0.85)
ax3.plot(df_hl_sorted['target_year'], df_hl_sorted['lag_to_peak'],
         color='gray', lw=1, ls='--', alpha=0.5)

for _, row in df_hl_sorted.iterrows():
    ax3.annotate(f"{int(row['lag_to_peak'])}y",
                 (row['target_year'], row['lag_to_peak']),
                 textcoords="offset points", xytext=(5, 4), fontsize=7)

ax3.set_title("C. Lag to Peak: How Many Years After an Event\nDoes Wikipedia Most Discuss It?",
              fontweight='bold')
ax3.set_xlabel("Target year (historical event)")
ax3.set_ylabel("Years between event and peak Wikipedia frequency")
ax3.text(0.02, 0.97,
         "Shorter lag = Wikipedia reacts faster to events\n(expected: post-2001 events have shorter lag)",
         transform=ax3.transAxes, fontsize=7.5, va='top',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

# ── Panel D: Heatmap of memory over time ─────────────────────
ax4 = fig.add_subplot(gs[1, 1])

# Pivot: rows = target years, columns = obs years
pivot = df_mem.pivot_table(
    index='target_year', columns='obs_year',
    values='rel_freq_10k', aggfunc='mean'
).fillna(0)

im = ax4.imshow(pivot.values, aspect='auto', cmap='YlOrRd',
                interpolation='nearest')
ax4.set_xticks(range(len(pivot.columns)))
ax4.set_xticklabels(pivot.columns.astype(str), rotation=90, fontsize=6.5)
ax4.set_yticks(range(len(pivot.index)))
ax4.set_yticklabels(pivot.index.astype(str), fontsize=7)
plt.colorbar(im, ax=ax4, label='Freq per 10k tokens', shrink=0.8)

ax4.set_title("D. Memory Heatmap: Which Years Are Discussed\nWhen (freq per 10k tokens)",
              fontweight='bold')
ax4.set_xlabel("Wikipedia snapshot year")
ax4.set_ylabel("Historical year being remembered")

# Main title
fig.suptitle(
    "Method 3 — Collective Memory in Wikipedia (2001–2024)\n"
    "Replication of Michel et al. (2011) Figure 3A on an alternative corpus",
    fontsize=13, fontweight='bold', y=1.01, color=SECONDARY
)

plt.savefig(OUTPUT_DIR / "method3_collective_memory.png", dpi=160,
            bbox_inches='tight', facecolor='white')
plt.show()
print("Figure saved to:", OUTPUT_DIR / "method3_collective_memory.png")

## 7. Key Findings for the Essay

Run this cell last. Copy these numbers directly into your essay.


In [ ]:
print("=" * 65)
print("METHOD 3 — KEY FINDINGS FOR ESSAY")
print("=" * 65)

print("\n1. HALF-LIFE RESULTS (Wikipedia vs. Michel et al.):")
print(f"{'Year':<8} {'Half-life (Wikipedia)':<25} {'Michel et al. ref'}")
print("-" * 50)
michel_ref = {1880: "~32 yrs", 1973: "~10 yrs"}
for _, row in df_hl.iterrows():
    ty  = int(row['target_year'])
    hl  = f"{int(row['half_life_yrs'])} yrs" if row['half_life_yrs'] else "NOT REACHED"
    ref = michel_ref.get(ty, "—")
    print(f"{ty:<8} {hl:<25} {ref}")

# Count how many years NEVER decayed
not_reached = df_hl[df_hl['half_life_yrs'].isna()]
print(f"\n  ⚠️  {len(not_reached)} out of {len(df_hl)} target years")
print(f"     never dropped below 50% of peak in our window.")
print(f"     This suggests Wikipedia's living-document nature")
print(f"     may RESIST the forgetting pattern Michel et al. found in books.")

print("\n2. LAG TO PEAK (years from event to maximum Wikipedia attention):")
for _, row in df_hl.sort_values('target_year').iterrows():
    ty  = int(row['target_year'])
    lag = int(row['lag_to_peak'])
    print(f"  {ty}: peak {lag} years after the event "
          f"(observation year {int(row['peak_obs_year'])})")

print("\n3. THEORETICAL INTERPRETATION FOR ESSAY:")
print("  Michel et al. found passive forgetting in static print corpora.")
print("  Wikipedia data shows one of two patterns (fill in from your results):")
print("  Pattern A: Forgetting IS present but slower — Wikipedia preserves memory better.")
print("  Pattern B: Forgetting is ABSENT — Wikipedia's continuous revision keeps")
print("             historical dates stably present rather than decaying.")
print("\n  Either result is publishable. Pattern A = replication with attenuation.")
print("  Pattern B = genuine divergence, supporting the 'living document' hypothesis.")

df_mem.to_csv(OUTPUT_DIR / "method3_memory_table.csv", index=False)
df_hl.to_csv(OUTPUT_DIR / "method3_halflife.csv", index=False)
print(f"\nData saved to: {OUTPUT_DIR}/")